In this module, we extract some insights on common subjects and their examination characteristics, based on past examination data. 
All data is sourced from https://www.vcaa.vic.edu.au/. We only use previous examination assessment reports in this notebook, as they contain the most interesting data.

In [1]:
# Resolve path
from pathlib import Path
import sys

parent_dir = str(Path().resolve().parents[0])
sys.path.insert(0, parent_dir)

In [2]:
import pandas as pd

from src.Qlassifier import api
#from nb_helpers import mcq_merged

In [3]:
# Set the scope: popular VCE subjects with large exam component
subjects = [
    "specialist_mathematics",
    "mathematical_methods",
    "physics",
    "chemistry"
]

years = range(2017, 2024) # Set timeframe

all_reports = api.process_material(subjects, years, type="report")

assert(all_reports)

In [4]:
all_reports["chemistry"]["2023"][0].head(5) # Example: 2023 examination 2 specialist math MCQ table

,question,correct_answer,%a,%b,%c,%d,comments
0,1.,B,8,67,8,17,"In human cells, glucose reacts with oxygen in ..."
1,2.,A,40,5,42,13,Fuel cells and galvanic cells both produce hea...
2,3.,A,70,2,26,1,The larger the number of C=C double bonds in t...
3,4.,D,14,2,5,79,The polarity of the physical electrodes does n...
4,5,D,16,38,3,42,All three statements are properties of coenzym...


There are some cool statistics & facts we can extract from this data. Below are but just a few of these which we will answer in this notebook:

- Do some subjects have a bias towards certain MCQ responses? Perhaps Chemistry exam writers are fascinated with making A) the correct answer.
- Which subjects have the highest frequency of "tricky" MCQ questions? (where the majority answer != correct answer).
- Which subjects have the "hardest" short-answer questions (and what does it mean for a question to be hard)?
 

In [5]:
all_reports["specialist_mathematics"]["2017_2"][0]

,question,%a,%b,%c,%d,%e,comments,correct_answer
0,1,1,8,75,4,12,"1 1\nx − ≤ ≤ ⇒ ∈ −∞ − ∪ ∞\n1\nx\n( , 1] [1, )",NaN
1,2,9,30,11,12,37,The solve and graphing capabilities of\na CAS ...,NaN
2,3,4,4,9,47,35,Use of complex solve gives five\nsolutions.,NaN
3,4,14,7,11,15,53,None,NaN
4,5,75,6,5,10,4,None,NaN
5,6,6,46,10,30,8,2 2\ndy d e\nx\n= =\narctan( )\ny\n(\n)\ndx dx...,NaN
6,7,4,6,20,60,9,None,NaN
7,8,4,29,7,52,7,3 m\n′′ =− ≥\n≥\nfx x m\n( ) 6 2 0 when\nx,NaN
8,9,4,45,12,9,30,None,NaN
9,10,31,9,45,7,6,f x ′′( ) does not change sign at a.,NaN


In [6]:
def mcq_merged(
    all_reports: dict[str, dict[str, list[pd.DataFrame]]], 
    years: list[int],
) -> dict[str, pd.DataFrame]:
    """ Returns a dictionary mapping each subject to one dataframe consisting
    of all multiple choice tables merged over the years indicated by years.
    """
    all_merged = {}
    subjects = all_reports.keys()
    for subject in subjects:
        report_types = list(all_reports[subject].keys())
        if "math" in subject:
            # only examination 2's have mcq sections
            dfs_to_merge = [all_reports[subject][report][0] for report in report_types \
                            if any(str(year) in report for year in years) and "_2" in report]
        else: 
            dfs_to_merge = [all_reports[subject][report][0] for report in report_types \
                            if any(str(year) in report for year in years)]
        
        merged = pd.concat(dfs_to_merge, axis=0, ignore_index=True)
        all_merged[subject] = merged
    return all_merged

mcqs = mcq_merged(all_reports, years = years)

spec_mcq = mcqs[subjects[0]]
meth_mcq = mcqs[subjects[1]]
phys_mcq = mcqs[subjects[2]]
chem_mcq = mcqs[subjects[3]]

Let's first try finding the proportion of correct MCQ options over the years 2020-23 (years where we have reliable data) for each subject. 

In [19]:
tmp = mcqs["specialist_mathematics"].dropna(subset = "correct_answer").reset_index(drop=True)
dum = tmp["correct_answer"].unique()
dum.sort()
dum

array(['', 'A', 'B', 'C', 'D', 'E'], dtype=object)

In [ ]:
for subject in subjects:
    # keep only data where we have the correct answer
    tmp = mcqs[subject].dropna(subset = "correct_answer").reset_index(drop=True)
    nrows = tmp.shape[0]

    # let's get the proporitons
    options = tmp["correct_answer"].unique()
    options.sort()

    proportions = []
    for option in options:
        proportions.append()
    num_a = sum(tmp["correct_answer"] == "A")
    num_a = sum(tmp["correct_answer"] == "A")


,question,%a,%b,%c,%d,%e,comments,correct_answer
0,1,4,1,92,2,1,None,NaN
1,2,1,5,12,80,1,None,NaN
2,3,2,2,83,12,1,None,NaN
3,4,3,10,6,7,75,None,NaN
4,5,47,20,9,16,7,"95% confidence interval is (0.039, 0.121).\nTh...",NaN
...,...,...,...,...,...,...,...,...
135,16,31,11,32,9,15,", \n, \n,",A
136,17,9,18,57,10,5,,C
137,18,8,23,22,39,6,and \n will have a maximum of three solutions ...,D
138,19,12,13,19,20,35,The graph of f is continuous over the interval...,E
